# 02 · SQL Analysis (PostgreSQL)
Answers 10 business questions directly against the `tmdb` PostgreSQL database built in notebook 01.
All SQL text lives in `queries.py` so the Streamlit app runs exactly the same queries.

Techniques covered: multi-table JOINs, GROUP BY, HAVING, subqueries, STRING_AGG, filtering, NULLIF for TMDB's "0 = unknown" rule.

In [1]:
import pandas as pd
from sqlalchemy import text
from db_config import get_engine
from queries import QUERIES, DEFAULT_PARAMS

engine = get_engine()
pd.set_option("display.max_rows", 30)

def sql(q, **params):
    """Run a SQL string against PostgreSQL and return a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(q), conn, params=params or None)

# quick connectivity check
sql("SELECT COUNT(*) AS movies FROM movies")

,movies
0,2503


## Q1 - Action movies released after 2015
*Recent Action slate*

In [2]:
# SQL shown for readability – identical to queries.py
q = '''
        SELECT m.title, m.release_date
        FROM movies m
        JOIN movie_genres mg ON mg.movie_id = m.movie_id
        JOIN genres g        ON g.genre_id  = mg.genre_id
        WHERE g.genre_name = 'Action'
          AND m.release_date > DATE '2015-12-31'
        ORDER BY m.release_date;
'''
df = sql(q)
print(len(df), 'rows')
df.head(15)

268 rows


,title,release_date
0,The 5th Wave,2016-01-14
1,Kung Fu Panda 3,2016-01-23
2,Zoolander 2,2016-02-06
3,Deadpool,2016-02-09
4,Triple 9,2016-02-19
5,Grimsby,2016-02-24
6,Gods of Egypt,2016-02-25
7,They Call Me Jeeg,2016-02-25
8,London Has Fallen,2016-03-02
9,Allegiant,2016-03-09


In [3]:
# auto-generated fact from the result
insight = f"{len(df)} Action titles released after 2015; peak year: {pd.to_datetime(df.release_date).dt.year.mode()[0] if len(df) else 'n/a'}"
print(insight)

268 Action titles released after 2015; peak year: 2016


**Insight:** Action remains an active post-2015 category. **Recommendation:** use this list as the benchmark set when evaluating new Action acquisitions (budget, rating, revenue comps).

## Q2 - Top 10 highest-grossing movies with genres
*Top grossers*

In [4]:
# SQL shown for readability – identical to queries.py
q = '''
        SELECT m.title, m.revenue,
               STRING_AGG(g.genre_name, ', ' ORDER BY g.genre_name) AS genres
        FROM movies m
        LEFT JOIN movie_genres mg ON mg.movie_id = m.movie_id
        LEFT JOIN genres g        ON g.genre_id  = mg.genre_id
        WHERE m.revenue > 0
        GROUP BY m.movie_id, m.title, m.revenue
        ORDER BY m.revenue DESC
        LIMIT 10;
'''
df = sql(q)
print(len(df), 'rows')
df.head(15)

10 rows


,title,revenue,genres
0,Avatar,2923706026,"Action, Adventure, Science Fiction"
1,Avengers: Endgame,2799439100,"Action, Adventure, Science Fiction"
2,Avatar: The Way of Water,2334484620,"Action, Adventure, Science Fiction"
3,Titanic,2264162353,"Drama, Romance"
4,Star Wars: The Force Awakens,2068223624,"Action, Adventure, Science Fiction"
5,Avengers: Infinity War,2052415039,"Action, Adventure, Science Fiction"
6,Spider-Man: No Way Home,1921206586,"Action, Adventure, Science Fiction"
7,Zootopia 2,1868208796,"Adventure, Animation, Comedy, Family, Mystery"
8,Inside Out 2,1698863816,"Adventure, Animation, Comedy, Family"
9,Jurassic World,1671537444,"Adventure, Science Fiction, Thriller"


In [5]:
# auto-generated fact from the result
insight = f"#1 {df.title[0]} grossed ${df.revenue[0]/1e6:,.0f}M; top 10 total ${df.revenue.sum()/1e9:,.2f}B"
print(insight)

#1 Avatar grossed $2,924M; top 10 total $21.60B


**Insight:** 9 of the top 10 grossers include Adventure: 6 are Action/Adventure/Science-Fiction blends (Avatar $2.92B, Avengers: Endgame $2.80B, Avatar: The Way of Water $2.33B …) and 2 are Animation/Family hits (Zootopia 2, Inside Out 2). Titanic (Drama/Romance) is the only exception. The top 10 alone grossed $21.6B. **Recommendation:** tentpole investment should target Adventure-led franchises – Action/Sci-Fi or Animation/Family – with broad audience reach.

## Q3 - Average budget & revenue by genre


In [6]:
# SQL shown for readability – identical to queries.py
q = '''
        -- zeros = unknown in TMDB, so they are excluded via NULLIF
        SELECT g.genre_name,
               ROUND(AVG(NULLIF(m.budget, 0)))  AS avg_budget,
               ROUND(AVG(NULLIF(m.revenue, 0))) AS avg_revenue,
               COUNT(*)                          AS movie_count
        FROM genres g
        JOIN movie_genres mg ON mg.genre_id = g.genre_id
        JOIN movies m        ON m.movie_id  = mg.movie_id
        GROUP BY g.genre_name
        ORDER BY avg_revenue DESC NULLS LAST;
'''
df = sql(q)
print(len(df), 'rows')
df.head(15)

19 rows


,genre_name,avg_budget,avg_revenue,movie_count
0,Adventure,110673598.0,400311337.0,674
1,Animation,87635508.0,371835705.0,246
2,Family,87906297.0,353369110.0,344
3,Science Fiction,96722246.0,333648844.0,463
4,Action,94227513.0,308453178.0,808
5,Fantasy,88640066.0,302705993.0,396
6,Comedy,54500177.0,221526374.0,762
7,Music,42817021.0,201566173.0,49
8,Romance,36939703.0,174811036.0,349
9,Thriller,51214051.0,168004245.0,673


In [7]:
# auto-generated fact from the result
insight = f"Highest avg revenue: {df.genre_name[0]} (${df.avg_revenue[0]/1e6:,.0f}M); lowest: {df.genre_name.iloc[-1]}"
print(insight)

Highest avg revenue: Adventure ($400M); lowest: Documentary


**Insight:** Adventure has the highest average revenue (≈$400M on a ≈$111M budget), followed by Animation ($372M) and Family ($353M); Documentary is lowest. Animation and Family earn almost as much as Adventure on ~20% smaller budgets. **Recommendation:** compare the revenue/budget *ratio*, not absolute revenue, when allocating capital – Animation/Family give blockbuster-level returns at lower cost.

## Q4 - Top 10 actors by movie count
*Most prolific actors*

In [8]:
# SQL shown for readability – identical to queries.py
q = '''
        SELECT actor_name, COUNT(DISTINCT movie_id) AS movie_count
        FROM "cast"
        GROUP BY person_id, actor_name
        ORDER BY movie_count DESC, actor_name
        LIMIT 10;
'''
df = sql(q)
print(len(df), 'rows')
df.head(15)

10 rows


,actor_name,movie_count
0,Samuel L. Jackson,39
1,Brad Pitt,38
2,Robert De Niro,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Mark Wahlberg,32
6,Scarlett Johansson,32
7,Willem Dafoe,32
8,Morgan Freeman,31
9,Tom Cruise,31


In [9]:
# auto-generated fact from the result
insight = f"{df.actor_name[0]} leads with {df.movie_count[0]} movies"
print(insight)

Samuel L. Jackson leads with 39 movies


**Insight:** Samuel L. Jackson leads with 39 films, followed by Brad Pitt and Robert De Niro (38 each). **Recommendation:** these are the most bankable/reliable casting names – but check their average rating too (Pandas Q8: Tom Hanks 7.29 vs Mark Wahlberg 6.63).

## Q5 - Directors with more than 3 movies
*Prolific directors*

In [10]:
# SQL shown for readability – identical to queries.py
q = '''
        SELECT person_name AS director, COUNT(DISTINCT movie_id) AS movie_count
        FROM crew
        WHERE job = 'Director'
        GROUP BY person_id, person_name
        HAVING COUNT(DISTINCT movie_id) > 3
        ORDER BY movie_count DESC, director;
'''
df = sql(q)
print(len(df), 'rows')
df.head(15)

217 rows


,director,movie_count
0,Steven Spielberg,27
1,Ridley Scott,19
2,Tim Burton,19
3,Martin Scorsese,16
4,Michael Bay,15
5,Robert Zemeckis,15
6,Clint Eastwood,14
7,M. Night Shyamalan,13
8,Robert Rodriguez,13
9,Antoine Fuqua,12


In [11]:
# auto-generated fact from the result
insight = f"{len(df)} directors have >3 films; top: {df.director[0] if len(df) else 'n/a'}"
print(insight)

217 directors have >3 films; top: Steven Spielberg


**Insight:** 217 directors have more than 3 films; Steven Spielberg leads with 27, then Ridley Scott and Tim Burton (19). **Recommendation:** directors with a proven multi-film record reduce production risk and are preferred for high-budget projects.

## Q6 - Top 10 most used keywords
*Popular themes*

In [12]:
# SQL shown for readability – identical to queries.py
q = '''
        SELECT keyword_name, COUNT(*) AS usage_count
        FROM movie_keywords
        GROUP BY keyword_id, keyword_name
        ORDER BY usage_count DESC
        LIMIT 10;
'''
df = sql(q)
print(len(df), 'rows')
df.head(15)

10 rows


,keyword_name,usage_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


In [13]:
# auto-generated fact from the result
insight = f"Top keyword: '{df.keyword_name[0]}' used {df.usage_count[0]} times"
print(insight)

Top keyword: 'based on novel or book' used 449 times


**Insight:** 'based on novel or book' (449) and 'sequel' (416) are the most-used tags – the market leans heavily on existing IP. Post-credit scenes (during/after-credits stingers) are also very common, a franchise-building tactic. **Recommendation:** prioritise adaptations and sequels in acquisition, and use these tags in recommendation engines and campaign copy.

## Q7 - Above-avg budget but below-avg revenue
*Expensive under-performers*

In [14]:
# SQL shown for readability – identical to queries.py
q = '''
        SELECT title, budget, revenue
        FROM movies
        WHERE budget  > (SELECT AVG(budget)  FROM movies WHERE budget  > 0)
          AND revenue > 0
          AND revenue < (SELECT AVG(revenue) FROM movies WHERE revenue > 0)
        ORDER BY budget - revenue DESC;
'''
df = sql(q)
print(len(df), 'rows')
df.head(15)

273 rows


,title,budget,revenue
0,Wake Up Dead Man: A Knives Out Mystery,210000000,4000000
1,The Gray Man,200000000,454023
2,The Tomorrow War,200000000,14400000
3,The Killer,175000000,362113
4,Red Notice,160000000,178143
5,The Irishman,159000000,968853
6,Turning Red,175000000,21813358
7,Luca,200000000,51074773
8,Mulan,200000000,69965374
9,Frankenstein,120000000,480678


In [15]:
# auto-generated fact from the result
insight = f"{len(df)} movies had above-avg budget but below-avg revenue"
print(insight)

273 movies had above-avg budget but below-avg revenue


**Insight:** 273 movies had above-average budgets but below-average revenue. Many are streaming-first releases with limited theatrical runs (e.g. The Gray Man, Red Notice, The Irishman), so low box office does not always mean a flop. **Recommendation:** add a stage-gate review for above-average budgets, and judge streaming titles on viewership, not box office.

## Q8 - Actors in movies by a director (Christopher Nolan)
*Director's recurring cast*

In [16]:
# SQL shown for readability – identical to queries.py
q = '''
        SELECT DISTINCT c.actor_name, m.title
        FROM crew d
        JOIN "cast" c ON c.movie_id = d.movie_id
        JOIN movies m ON m.movie_id = d.movie_id
        WHERE d.job = 'Director' AND d.person_name = :director
        ORDER BY m.title, c.actor_name;
'''
df = sql(q, **DEFAULT_PARAMS)
print(len(df), 'rows')
df.head(15)

110 rows


,actor_name,title
0,Christian Bale,Batman Begins
1,Cillian Murphy,Batman Begins
2,Gary Oldman,Batman Begins
3,Katie Holmes,Batman Begins
4,Ken Watanabe,Batman Begins
5,Liam Neeson,Batman Begins
6,Mark Boone Junior,Batman Begins
7,Michael Caine,Batman Begins
8,Rutger Hauer,Batman Begins
9,Tom Wilkinson,Batman Begins


In [17]:
# auto-generated fact from the result
insight = f"{df.actor_name.nunique()} distinct actors across {df.title.nunique()} films of the director"
print(insight)

84 distinct actors across 11 films of the director


**Insight:** Christopher Nolan's 11 films use 84 distinct actors, with clear repeat collaborators (Michael Caine, Cillian Murphy, Christian Bale). **Recommendation:** when signing a director, factor in their preferred cast (package deals).

## Q9 - Highest-rated genre (min 20 movies)
*Best-rated genre*

In [18]:
# SQL shown for readability – identical to queries.py
q = '''
        SELECT g.genre_name,
               ROUND(AVG(m.vote_average)::numeric, 2) AS avg_rating,
               COUNT(*) AS movie_count
        FROM genres g
        JOIN movie_genres mg ON mg.genre_id = g.genre_id
        JOIN movies m        ON m.movie_id  = mg.movie_id
        WHERE m.vote_count > 0
        GROUP BY g.genre_name
        HAVING COUNT(*) >= 20
        ORDER BY avg_rating DESC;
'''
df = sql(q)
print(len(df), 'rows')
df.head(15)

17 rows


,genre_name,avg_rating,movie_count
0,War,7.36,85
1,History,7.34,112
2,Music,7.32,49
3,Drama,7.25,937
4,Animation,7.20,246
5,Western,7.12,33
6,Romance,7.03,349
7,Crime,7.01,387
8,Family,6.93,344
9,Mystery,6.88,216


In [19]:
# auto-generated fact from the result
insight = f"Highest-rated genre (>=20 films): {df.genre_name[0]} at {df.avg_rating[0]}"
print(insight)

Highest-rated genre (>=20 films): War at 7.36


**Insight:** War (7.36), History (7.34) and Music (7.32) are the highest-rated genres, while Adventure – the top revenue genre – rates only 6.83. Critical acclaim and box office do not coincide. **Recommendation:** high-rated genres suit prestige/awards and streaming libraries; high-revenue genres suit theatrical.

## Q10 - Top 10 movies by keyword count
*Richly tagged movies*

In [20]:
# SQL shown for readability – identical to queries.py
q = '''
        SELECT m.title, COUNT(mk.keyword_id) AS keyword_count
        FROM movies m
        JOIN movie_keywords mk ON mk.movie_id = m.movie_id
        GROUP BY m.movie_id, m.title
        ORDER BY keyword_count DESC, m.title
        LIMIT 10;
'''
df = sql(q)
print(len(df), 'rows')
df.head(15)

10 rows


,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,13 Hours: The Secret Soldiers of Benghazi,49
9,How to Train Your Dragon: The Hidden World,49


In [21]:
# auto-generated fact from the result
insight = f"Max keywords on one movie: {df.keyword_count[0]}"
print(insight)

Max keywords on one movie: 98


**Insight:** Smile 2 carries 98 keywords and Taken 3 70, far above typical titles. Richly tagged movies are easier to recommend and search. **Recommendation:** enforce a minimum keyword set for every title in the catalogue to improve discoverability.

## Summary
The SQL layer answers who (actors/directors), what (genres/keywords) and how much (budget/revenue) questions. Key takeaways feed the README top-5 insights: genre economics (Q3), expensive flops (Q7) and rating-vs-revenue split by genre (Q9 vs Q3).